# Homework 1

_**[Power Systems Optimization](https://github.com/east-winds/power-systems-optimization)**_

_by Jesse D. Jenkins and Michael R. Davidson (last updated: September 14, 2022)_

This Notebook will walk you through defining a simple transport flow model and then ask you to interact with the solutions and modify to model to add additional constraints...

## Setting up the model

### Load packages

In [46]:
using JuMP
using HiGHS
using DataFrames
using CSV

### Define sets

We will define two sets, both as arrays of strings

***Production plants, $P$***

In [47]:
P=["trenton", "newark"] # production plants

2-element Vector{String}:
 "trenton"
 "newark"

***Markets for products, $M$***

In [48]:
M=["newyork", "princeton", "philadelphia"] # markets for products

3-element Vector{String}:
 "newyork"
 "princeton"
 "philadelphia"

Note that sets can also be defined over intervals (as in `i=1:10`) or numerical vectors (as in `x=[2, 4, 5, 11]`) 

### Define parameters

We'll make use of the defined sets as indexes for our parameters...

***Plant production capacities***

In [49]:
plants = DataFrame(plant=P, capacity=[350,650])

Row,plant,capacity
,String,Int64
1,trenton,350
2,newark,650


***Demand for products***

Stored in a [DataFrame](https://juliadata.github.io/DataFrames.jl/stable/)

In [50]:
markets = DataFrame(
    market=M, 
    demand=[325, 300, 275]
)

Row,market,demand
,String,Int64
1,newyork,325
2,princeton,300
3,philadelphia,275


A few different ways to index into our DataFrames to access parameters (all of the below are equivalent)

In [51]:
plants[plants.plant.=="newark",:capacity] # option 1

1-element Vector{Int64}:
 650

In [52]:
plants[plants.plant.=="newark",:].capacity # option 2

1-element Vector{Int64}:
 650

In [53]:
plants.capacity[plants.plant.=="newark"] # option 3

1-element Vector{Int64}:
 650

In [54]:
plants[:,:capacity][plants.plant.=="newark"] # option 4

1-element Vector{Int64}:
 650

Note that DataFrame indexing returns an Array by default, in this case, a 1-element Array of type Int64 (64-bit integer), as indicated by `Array{Int64,1}` above. 

To access the single Int64 value, append `[1]` to any of the above to reference the first (and only) element in this array. 

In [55]:
plants.capacity[plants.plant.=="newark"][1]

650

In [56]:
typeof(plants.capacity[plants.plant.=="newark"][1])

Int64

In [57]:
typeof(plants.capacity[plants.plant.=="newark"])

Vector{Int64} (alias for Array{Int64, 1})

***Distance from plants to markets***

Stored in a JuMP [DenseAxisArray](https://jump.dev/JuMP.jl/v0.19/containers/) with data array and symbolic references across each of our sets (plants and markets), converted to Symbols for referencing

In [58]:
# two dimensional symbolic DenseAxisArray, with from/to distance pairs
distances = JuMP.Containers.DenseAxisArray(
    [2.5 0.5 1.5;
     0.5 1.5 3.5],
    Symbol.(P),
    Symbol.(M),
)

2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [:trenton, :newark]
    Dimension 2, [:newyork, :princeton, :philadelphia]
And data, a 2×3 Matrix{Float64}:
 2.5  0.5  1.5
 0.5  1.5  3.5

A couple example references to our DenseAxisArray to access parameters...

In [59]:
distances[:trenton, :newyork] #example of distance references

2.5

In [60]:
distances[:newark, :newyork] #example of distance references

0.5

In [61]:
distances[Symbol(P[2]),Symbol(M[1])] # another way to find distance from newark to trenton

0.5

In [62]:
distances[Symbol("newark"), Symbol("newyork")] # and a third...

0.5

***Costs of transport***

In [63]:
freight_cost = 90 # Cost of freight shipment per unit of distance

90

### Create model
(and specify the HiGHS solver)

In [64]:
transport = Model(HiGHS.Optimizer);

### Define variables

***Quantities of product to transport from plant $p \in P$ to market $m \in M$***

In [65]:
@variable(transport, X[P,M] >= 0)

2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, ["trenton", "newark"]
    Dimension 2, ["newyork", "princeton", "philadelphia"]
And data, a 2×3 Matrix{VariableRef}:
 X[trenton,newyork]  X[trenton,princeton]  X[trenton,philadelphia]
 X[newark,newyork]   X[newark,princeton]   X[newark,philadelphia]

Example reference to single quantity decision variable, the quantity shipped from Newark to Philadelphia:

In [66]:
X["newark","philadelphia"]

X[newark,philadelphia]

### Define constraints

***Supply capacity constraint***

In [67]:
@constraint(transport, cSupply[p in P], 
    sum(X[p,m] for m in M) 
    <= plants.capacity[plants.plant.==p][1])

1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, ["trenton", "newark"]
And data, a 2-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 cSupply[trenton] : X[trenton,newyork] + X[trenton,princeton] + X[trenton,philadelphia] ≤ 350
 cSupply[newark] : X[newark,newyork] + X[newark,princeton] + X[newark,philadelphia] ≤ 650

***Demand balance constraint***

Ensure all demand is satisfied at each market

In [68]:
@constraint(transport, cDemand[m in M], 
    sum(X[p,m] for p in P) 
    >= markets.demand[markets.market.==m][1])

1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, ["newyork", "princeton", "philadelphia"]
And data, a 3-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}}:
 cDemand[newyork] : X[trenton,newyork] + X[newark,newyork] ≥ 325
 cDemand[princeton] : X[trenton,princeton] + X[newark,princeton] ≥ 300
 cDemand[philadelphia] : X[trenton,philadelphia] + X[newark,philadelphia] ≥ 275

### Define objective function

Minimize total cost of transport to satisfy all demand.

First we'll define an expression for total cost of shipments...

In [69]:
@expression(transport, eCost, 
    sum(freight_cost*distances[Symbol(p),Symbol(m)]*X[p,m] 
        for p in P, m in M)
    )

225 X[trenton,newyork] + 45 X[trenton,princeton] + 135 X[trenton,philadelphia] + 45 X[newark,newyork] + 135 X[newark,princeton] + 315 X[newark,philadelphia]

Now we'll minimize this total cost

In [70]:
@objective(transport, Min, eCost)

225 X[trenton,newyork] + 45 X[trenton,princeton] + 135 X[trenton,philadelphia] + 45 X[newark,newyork] + 135 X[newark,princeton] + 315 X[newark,philadelphia]

## Interact with the model

**(a)** Now let's solve the model. In the blank cell below, enter the command for JuMP to solve a model and run the cell

In [71]:
print(transport)
optimize!(transport)

Running HiGHS 1.13.1 (git hash: 1d267d97c): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
LP has 5 rows; 6 cols; 12 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e+02, 6e+02]
Presolving model
5 rows, 6 cols, 12 nonzeros  0s
5 rows, 6 cols, 12 nonzeros  0s
Presolve reductions: rows 5(-0); columns 6(-0); nonzeros 12(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 3(900) 0.0s
          4     8.5500000000e+04 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 4
Objective value     :  8.5500000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


**(b)** You've got a solution. Now query the objective function in the empty cell below and save it to a variable (name of your choice)

In [72]:
obj_function = @objective(transport, Min, eCost)
print(obj_function)

225 X[trenton,newyork] + 45 X[trenton,princeton] + 135 X[trenton,philadelphia] + 45 X[newark,newyork] + 135 X[newark,princeton] + 315 X[newark,philadelphia]

**(c)** Now query and save the optimal solution for X (the decisions about shipment quantities from plant to market) to an Array or DataFrame

In [73]:
decision = value.(X)

┌ Warning: The model has been modified since the last call to `optimize!` (or `optimize!` has not been called yet). If you are iteratively querying solution information and modifying a model, query all the results first, then modify the model.
└ @ JuMP ~/.julia/packages/JuMP/0tD10/src/optimizer_interface.jl:1232


LoadError: OptimizeNotCalled()

**(d)** Please interpret your results by writing an explanation in the markdown cell below. 

Which facility or facilities supplies the most demand in New York? Does this result make sense? Why?

Which facility or facilities supplies the most demand in Philadelphia? Does this result make sense? Why?

Which facility or facilities supplies the demand in Princeton? Does this result make sense? Why?

Trenton supplies the most demand in New York. This makes sense because the distance between New York and Trenton is less than New York to Newark and we are trying to minimize cost of transport. The distance is also the least so the model prioritized filling this demand from the Trenton supply first.

Trenton supplies the most demand in Philadelphia. This makes sense because the distance between Newark and Philadelphia is less than the distance between Philadelphia and Trenton. This is also the shortest distance for the Newark supply to travel so the model priorizes filling this demand first. 

Newark supplies the most demand in Princeton. At first glance this seems to be the opposite because Trenton is much closer to Princeton than Newark, however the distance between Philadelphia and Newark is significantly greater than Princeton to Newark so the model avoids allowing any of the Philadelphia demand to come from Newark.

**(e)** A new market in New Brunswick appears, with a demand for 50 units. It is located 1.0 units away from both plants. Add this market to the model and solve again.

In [ ]:
M_new = ["newyork", "princeton", "philadelphia", "new brunswick"]

new_markets = DataFrame(
    new_market=M_new, 
    demand=[325, 300, 275, 50]
)
distances_new = JuMP.Containers.DenseAxisArray(
    [2.5 0.5 1.5 1.0;
     0.5 1.5 3.5 1.0],
    Symbol.(P),
    Symbol.(M_new),
)
transport_new = Model(HiGHS.Optimizer);
@variable(transport_new, X[P,M_new] >= 0)
@constraint(transport_new, cSupply[p in P], 
    sum(X[p,m] for m in M_new) 
    <= plants.capacity[plants.plant.==p][1])
@constraint(transport_new, cDemand[m in M_new], 
    sum(X[p,m] for p in P) 
    >= new_markets.demand[new_markets.new_market.==m][1])
@expression(transport_new, eCost, 
    sum(freight_cost*distances_new[Symbol(p),Symbol(m)]*X[p,m] 
        for p in P, m in M_new)
    )
@objective(transport_new, Min, eCost)


Running HiGHS 1.13.1 (git hash: 1d267d97c): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
LP has 6 rows; 8 cols; 16 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [5e+01, 6e+02]
Presolving model
6 rows, 8 cols, 16 nonzeros  0s
6 rows, 8 cols, 16 nonzeros  0s
Presolve reductions: rows 6(-0); columns 8(-0); nonzeros 16(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 4(950) 0.0s
          5     9.0000000000e+04 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 5
Objective value     :  9.0000000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


**(f)** What is new optimal solution? 

In [ ]:
print(transport_new)
optimize!(transport_new)    
new_decision = value.(X)

LP has 6 rows; 8 cols; 16 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [5e+01, 6e+02]
Solving LP with useful basis so presolve not used

Model status        : Optimal
Objective value     :  9.0000000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, ["trenton", "newark"]
    Dimension 2, ["newyork", "princeton", "philadelphia", "new brunswick"]
And data, a 2×4 Matrix{Float64}:
   0.0   75.0  275.0   0.0
 325.0  225.0    0.0  50.0

**(g)** Interpret this result in the markdown cell below. Which facility or facilities supplies the demand in New Brunswick? Does this result make sense? Why?

The demand in New Brunswick is met by the supply in Newark. This result makes sense because New Brunswick is much closer to Newark than Philidelpia is. The model still prioritizes avoiding demand in Philadelphia to be filled by supply in Newark.
